# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

# Reading from Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")


# Transformations

- Trim text columns
- Clean date columns 
    - Change date columns to same format of YYYY-mm-dd as other tables
    - Order date has 0s
- sales price column has negatives and NULLs
- sls_sales has negatives and NULLs
- Clean price column by calculating using sales and quantity when available and price is not null or <= 0
- Rename column headers

### Trimming text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Cleaning date columns

In [0]:
df = (
    df
     .withColumn(
         "sls_order_dt",
         F.when(
            (F.col("sls_order_dt") == 0) | (F.length(F.col("sls_order_dt")) != 8),
            None
         ).otherwise(F.to_date(F.col("sls_order_dt"), "yyyyMMdd"))
    ).withColumn(
        "sls_ship_dt",
        F.when(
            (F.col("sls_ship_dt") == 0) | (F.length(F.col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(F.col("sls_ship_dt"), "yyyyMMdd"))
    ).withColumn(
        "sls_due_dt",
        F.when(
            (F.col("sls_due_dt") == 0) | (F.length(F.col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(F.col("sls_due_dt"), "yyyyMMdd"))
    )
)

### Cleaning price column

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_sales") > 0) & (col("sls_quantity") > 0),
            col("sls_sales") / col("sls_quantity")
        )
         .when(
            (col("sls_sales") > 0) & (col("sls_price").isNull() | (col("sls_price") <= 0)),
            col("sls_sales") / col("sls_quantity")
        ).otherwise(None)
    )
)


### Renaming column headers

In [0]:
RENAME_MAPPING = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAPPING.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to silver table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.crm_sales")

### Sanity check silver table

In [0]:
%sql

SELECT * FROM workspace.silver.crm_sales